# 01 · Exploración del subcircuito olfativo (FlyWire FAFB v783)
Cadena ORN→PN→KC→MBON + neuronas locales (LN). Ejecutar desde la raíz del repo.

In [1]:
import os, sys; os.chdir('..') if os.path.basename(os.getcwd())=='notebooks' else None
sys.path.insert(0, 'src')
import pandas as pd
from flyconn.data.circuit import load_neurons, load_edges
pd.set_option('display.width', 160)
n = load_neurons(); e = load_edges(n)
print(len(n), 'neuronas;', len(e), 'aristas')

8668 neuronas; 131458 aristas


## Neuronas por capa y lado

In [2]:
pd.crosstab(n.layer, n.side.fillna('sin lado'), margins=True)

side,center,left,right,sin lado,All
layer,,,,,
KC,0,2580,2597,0,5177
LN,2,214,213,0,429
MBON,0,48,48,0,96
ORN,0,1117,1134,30,2281
PN,0,341,344,0,685
All,2,4300,4336,30,8668


## Neurotransmisor predicho por capa (y confianza media)

In [3]:
pd.crosstab(n.layer, n.nt_type, margins=True).join(n.groupby('layer').nt_type_score.mean().rename('conf_media'))

,ACH,GABA,GLUT,OCT,SER,All,conf_media
layer,,,,,,,
KC,5177,0,0,0,0,5177,1.000000
LN,43,113,97,2,24,279,0.341329
MBON,51,17,23,0,0,91,0.736562
ORN,1420,3,0,0,391,1814,0.500473
PN,474,203,8,0,0,685,0.907606
All,7165,336,128,2,415,8046,NaN


## Aristas y sinapsis por par de capas

In [4]:
e.groupby(['pre_layer','post_layer']).agg(aristas=('syn_count','size'), sinapsis=('syn_count','sum'), mediana=('syn_count','median'))

aristas  sinapsis  mediana
pre_layer post_layer                            
KC        MBON          26937    226932      7.0
LN        LN            13397    408132     14.0
          ORN           10034     88796      7.0
          PN            17055    304144     11.0
ORN       LN            16304    161876      7.0
          PN            12233    274396     14.0
PN        KC            22298    366946     15.0
          LN            13200    291418     13.0

## Conexiones entre hemisferios (lado pre ≠ lado post)

In [5]:
side = n.set_index('root_id').side
x = e.assign(pre_side=e.pre_root_id.map(side), post_side=e.post_root_id.map(side))
x['cruza'] = (x.pre_side != x.post_side)
x.groupby(['pre_layer','post_layer']).apply(lambda g: pd.Series({'aristas_cruzadas': int(g.cruza.sum()), 'frac_sinapsis_cruzadas': g.syn_count[g.cruza].sum()/g.syn_count.sum()}))

aristas_cruzadas  frac_sinapsis_cruzadas
pre_layer post_layer                                          
KC        MBON                  8128.0                0.376223
LN        LN                    1511.0                0.071467
          ORN                   3924.0                0.374352
          PN                    1840.0                0.099538
ORN       LN                   10005.0                0.573878
          PN                    4848.0                0.335734
PN        KC                     449.0                0.017667
          LN                     790.0                0.045673

## ORN: tipos de receptor y simetría entre lados

In [6]:
orn = n[n.layer=='ORN']
t = pd.crosstab(orn.primary_type.fillna('sin tipo'), orn.side.fillna('sin lado'))
print(len(t), 'tipos'); t[(t.get('left',0)==0)|(t.get('right',0)==0)|(t.index=='sin tipo')]

54 tipos


side,left,right,sin lado
primary_type,,,
sin tipo,0,2,0


## PN: ORN uniglomerulares — ¿cada tipo de ORN conecta con su tipo de PN?

In [7]:
ty = n.set_index('root_id').primary_type
oe = e[(e.pre_layer=='ORN')&(e.post_layer=='PN')]
oe = oe.assign(orn=oe.pre_root_id.map(ty), pn=oe.post_root_id.map(ty))
oe.groupby('orn').syn_count.sum().sort_values(ascending=False).head(10), oe.groupby('pn').syn_count.sum().sort_values(ascending=False).head(10)

(orn
 ORN_DP1m    19728
 ORN_DA1     12812
 ORN_V       10173
 ORN_DM1     10139
 ORN_DL2v     9669
 ORN_VA1v     9595
 ORN_VA1d     8358
 ORN_VM5d     7989
 ORN_VL2a     7789
 ORN_DM2      7324
 Name: syn_count, dtype: int64,
 pn
 DP1m_adPN    12504
 DA1_lPN      11596
 M_l2PNl20    11037
 DM1_lPN       8838
 DL2v_adPN     8803
 VM5d_adPN     8113
 DM2_lPN       7111
 DL1_adPN      7054
 VA2_adPN      6474
 VA6_adPN      6306
 Name: syn_count, dtype: int64)

## Grado de entrada de las KC (nº de PN por KC) y de salida a MBON

In [8]:
kc_in = e[e.post_layer=='KC'].groupby('post_root_id').size()
mb_in = e[e.post_layer=='MBON'].groupby('post_root_id').size()
kc_all = n[n.layer=='KC'].root_id
print('KC sin ninguna PN de entrada:', (~kc_all.isin(kc_in.index)).sum(), 'de', len(kc_all))
print(kc_in.describe()); print('MBON sin KC:', (~n[n.layer=='MBON'].root_id.isin(mb_in.index)).sum())

KC sin ninguna PN de entrada: 427 de 5177
count    4750.000000
mean        4.694316
std         1.616804
min         1.000000
25%         4.000000
50%         5.000000
75%         6.000000
max        10.000000
dtype: float64
MBON sin KC: 12


## Neuronas sin aristas dentro del subcircuito

In [9]:
used = set(e.pre_root_id)|set(e.post_root_id)
n.assign(conectada=n.root_id.isin(used)).groupby('layer').conectada.agg(['sum','size'])

,sum,size
layer,,
KC,5167,5177
LN,429,429
MBON,84,96
ORN,2279,2281
PN,642,685
